# `<TICKER>__L4_to_L9.ipynb` — per-asset runner

Runs the minimal per-asset pipeline from L4 materialization to L9 endproduct and writes the seven-file deliverable into `Assets/<TICKER>/`. `run_asset.py` injects the ticker, executes this notebook, and saves the executed copy as file #1.

## 00 — Setup

In [ ]:
import sys
from pathlib import Path

STRUCTURE_DIR = str(Path.cwd())
sys.path.insert(0, STRUCTURE_DIR)
import pipeline as P
import asset_writers as W

TICKER = "AAPL"
DB_PATH = P.bars_db()  # data/liora.duckdb (full) or data/bars_demo.duckdb (bundled demo)

ASSET_DIR = P.REPO_ROOT / "Assets" / TICKER
ASSET_DIR.mkdir(parents=True, exist_ok=True)
AF = {
    "notebook":   ASSET_DIR / f"{TICKER}__L4_to_L9.ipynb",
    "parquet":    ASSET_DIR / f"{TICKER}_ohlcv_1h.parquet",
    "parquet_1d": ASSET_DIR / f"{TICKER}_ohlcv_1d.parquet",
    "parquet_1w": ASSET_DIR / f"{TICKER}_ohlcv_1w.parquet",
    "optuna":     ASSET_DIR / "OPTUNAs_XGB_HPOs_best_params.json",
    "strategy":   ASSET_DIR / f"strategy_{TICKER}.py",
    "readme":     ASSET_DIR / f"{TICKER}_README.md",
}
P.validate_parameters()
SEED = P.seed_everything()
MANIFEST = P.resolve_feature_manifest(TICKER)
THRESH = P.PIPELINE_PARAMETERS["THRESHOLD_ENTRY"]
CAPITAL_MODE = P.PIPELINE_PARAMETERS["CAPITAL_MODE"]
print("TICKER", TICKER, "| DB", DB_PATH, "| features", MANIFEST["effective_feature_count"])
print("namespaces", MANIFEST["active_namespaces"])


## L4 — DuckDB snapshot → clean 1h parquet, then 1d / 1w roll-up

In [ ]:
df = P.layer4_snapshot_to_parquet(DB_PATH, TICKER, AF["parquet"])
print("L4 clean 1h rows:", len(df), "|", df["timestamp"].min(), "->", df["timestamp"].max())


In [ ]:
P.layer4_materialize_timeframes(df, {"1d": AF["parquet_1d"], "1w": AF["parquet_1w"]})
import pandas as pd
for tf in ("1d", "1w"):
    d = pd.read_parquet(AF[f"parquet_{tf}"])
    print(f"L4 {tf} rows:", len(d), "| cols:", list(d.columns))


## L5 → L6 — time split, feature namespaces, Triple Barrier Y

In [ ]:
REC = P.derive_output_b(df, TICKER, MANIFEST)
print("train candidates:", len(REC["train_events"]), "| Output-B rows:", len(REC["df_b"]))
print("bounds:", REC["bounds"])
print("audit:", REC["audit"])


## L7 — Optuna best hyperparameters + Kelly

In [ ]:
BEST, AP, FOLDS = P.layer7_optuna(REC["df_b"], REC["bounds"], SEED, MANIFEST)
print("best:", BEST)
print("cv AUC-PR:", AP, "| folds:", FOLDS)


In [ ]:
BEST["kelly_fraction"] = (P.calibrate_kelly(REC["df"], REC["df_b"], REC["train_events"], REC["bounds"], BEST, SEED, MANIFEST)
                          if CAPITAL_MODE.startswith("kelly") else None)
print("kelly_fraction (lambda):", BEST["kelly_fraction"], "| CAPITAL_MODE:", CAPITAL_MODE)


In [ ]:
import json
json.dump({"best_params": BEST, "feature_manifest": MANIFEST,
           "features": {"ids": MANIFEST["effective_feature_ids"],
                        "names": MANIFEST["effective_feature_names"],
                        "active_namespaces": MANIFEST["active_namespaces"],
                        "per_namespace": MANIFEST["per_namespace"]},
           "cv_auc_pr": AP, "cv_folds": FOLDS, "n_trials": P.n_trials()},
          open(AF["optuna"], "w"), indent=2)
print("wrote", AF["optuna"].name)


## L8 — XGBoost strategy artifact

In [ ]:
BOOSTER = P.layer8_train(REC["df_b"], BEST, SEED, MANIFEST)
KF = BEST.get("kelly_fraction") if CAPITAL_MODE.startswith("kelly") else None
tr_scored = P.score_setups(BOOSTER, REC["df"], REC["train_events"], MANIFEST, REC["feature_context"])
tr_summary, _, _ = P.run_engine(REC["df"], tr_scored, REC["bounds"]["train_start_idx"],
                                REC["bounds"]["oos_start_idx"] - 1, THRESH, kelly_fraction=KF)
ACCEPT = P.accept_strategy(tr_summary)
print("train acceptance:", ACCEPT)


In [ ]:
sp = P.PIPELINE_PARAMETERS["splits"]
META = P.strategy_meta(BOOSTER, REC["df_b"], TICKER, BEST, {},
                       {"start": sp["train_start"], "end": sp["train_end"]}, MANIFEST)
META["ACCEPTANCE"] = ACCEPT
W.write_strategy(AF["strategy"], META)
print("wrote", AF["strategy"].name)


## L9 — OOS verdict and endproduct

In [ ]:
oos_events, _ = P.generate_candidate_events(df, REC["masks"]["oos"], REC["feature_context"])
oos_scored = P.score_setups(BOOSTER, df, oos_events, MANIFEST, REC["feature_context"])
SUMM, LEDGER, _ = P.run_engine(df, oos_scored, REC["bounds"]["oos_start_idx"],
                               REC["bounds"]["oos_end_idx"], THRESH, kelly_fraction=KF)
print("OOS:", {k: SUMM[k] for k in ("start_capital", "end_capital", "profit_factor", "trades")})

if SUMM["trades"] == 0:
    SUMM, LEDGER, _ = P.hodl_fallback(df, REC["bounds"]["oos_start_idx"], REC["bounds"]["oos_end_idx"])
    print("OOS model trades = 0 -> HODL fallback:",
          {k: SUMM[k] for k in ("start_capital", "end_capital", "return_pct", "trades")})

In [ ]:
W.write_readme(AF["readme"], {**SUMM, "ticker": TICKER,
                              "capital_mode": SUMM.get("capital_mode", CAPITAL_MODE),
                              "kelly_fraction": None if SUMM.get("hodl_fallback") else BEST.get("kelly_fraction")},
               LEDGER, MANIFEST)
print("wrote", AF["readme"].name)

OOS_DB = Path(P.oos_db())  # data/oos_metrics.db (env OOS_METRICS_DB redirects it for make verify)
W.write_oos_metrics(OOS_DB, {**SUMM, "ticker": TICKER, "cv_auc_pr": AP, "cv_folds": FOLDS,
                             "oos_window": f'{sp["oos_start"]} -> {sp["oos_end"]}'})
print("L9 ->", OOS_DB.name, ":", TICKER)


In [ ]:
six = [AF["parquet"], AF["parquet_1d"], AF["parquet_1w"], AF["optuna"], AF["strategy"], AF["readme"]]
missing = [p.name for p in six if not p.exists()]
assert not missing, f"missing deliverables: {missing}"
print("6/7 deliverables present (notebook copy added by run_asset.py):")
print(sorted(p.name for p in ASSET_DIR.iterdir() if p.is_file()))
